# 🏥 SyftBox — Data Owner Tutorial

Welcome! This notebook walks you through using **SyftBox** as a **Data Owner** (DO).

## What is a Data Owner?

You hold private data that other people — **Data Scientists** (DS) — want to analyze. SyftBox lets them submit Python jobs that run *on your machine*, against your private data, returning only the results you choose to share back.

You stay in control:
- 🔒 The raw data never leaves your machine.
- ✅ You decide who can connect (peer approval).
- 📋 You review every job's source code before it runs (or whitelist patterns for auto-approval).
- 📤 Only the files in `outputs/` come back to the Data Scientist.

## What's in this tutorial

| Step | What you'll do |
|------|----------------|
| 1 | Install `syft-client` |
| 2 | Configure your DO email |
| 3 | One-time: convert OAuth credentials → drive token |
| 4 | Log in as Data Owner |
| 5 | Approve incoming peer requests |
| 6 | Create datasets — both a simple CSV and a CSV-with-PDFs example |
| 7 | List & inspect submitted jobs |
| 8 | Review and approve (or reject) a job |
| 9 | Run approved jobs and share results |
| 10 | Set up auto-approval for trusted job patterns |
| 11 | Run unattended with `syft_bg` background services |
| 12 | Inspect service logs |
| 13 | Cleanup — delete datasets, stop services |

## Before you start

You need:
1. **A Google account** — used as your SyftBox identity. Drive is the transport channel.
2. **OAuth client credentials** (`credentials.json` from Google Cloud Console). One-time setup; see Step 3.
3. **The Data Scientist's email** — they'll send you a peer request, and you decide whether to approve it.

This notebook is designed to run **locally** (laptop or server), not Colab — Data Owners typically need persistent processes for the background services in Step 11.


---
## Step 1 — Install `syft-client`

We pin a specific version so this tutorial keeps working as the library evolves.

In [ ]:
!uv pip install -q "syft-client==0.1.112" reportlab


---
## Step 2 — Configure

Set your Data Owner email. You can optionally pre-list the Data Scientists you trust (for use in Step 6 when sharing datasets).

In [ ]:
import syft_client as sc
sc.__version__


In [ ]:
# ✏️ EDIT THESE
DO_EMAIL    = "data.owner@example.com"      # ✏️ your SyftBox/Google email
DS_EMAILS   = ["data.scientist@example.com"]  # ✏️ Data Scientists you intend to share with

# ── Validation ──────────────────────────────────────────────────────────
issues = []
if DO_EMAIL == "data.owner@example.com" or "@" not in DO_EMAIL:
    issues.append(f"❌  DO_EMAIL ({DO_EMAIL!r}) is still the placeholder or invalid.")
for ds in DS_EMAILS:
    if "@" not in ds or ds == "data.scientist@example.com":
        issues.append(f"❌  DS email ({ds!r}) is still the placeholder or invalid.")

if issues:
    print("⚠️  Fix the following before continuing:\n")
    for msg in issues: print(" ", msg)
else:
    print("✅  Configuration looks good!")
    print(f"    DO email  : {DO_EMAIL}")
    print(f"    DS emails : {DS_EMAILS}")


---
## Step 3 — One-time: convert OAuth credentials to a drive token

Unlike Colab (which auto-detects your Google account), local Python needs an explicit OAuth token. This is **one-time setup**.

### What you need

1. A Google Cloud project with the **Drive API** enabled.
2. **OAuth 2.0 client credentials** — a `credentials.json` file you download from the GCP console.
   - Application type: **Desktop app**
   - Test users: include your DO email
3. The path to that file in `CREDENTIALS_PATH` below.

`sc.credentials_to_token(...)` opens a browser for you to authorize, then writes a `token_*.json` you'll reuse forever.

> 💡 Already have a token from a previous session? Set `TOKEN_PATH` directly and skip the conversion cell.

In [ ]:
from pathlib import Path

# ✏️ EDIT THESE PATHS
CREDENTIALS_PATH = Path("~/.syft-creds/credentials.json").expanduser()   # ✏️ from GCP console
TOKEN_PATH       = Path("~/.syft-creds/token_do.json").expanduser()       # ✏️ where to save the token

if TOKEN_PATH.exists():
    print(f"✅  Token already exists at {TOKEN_PATH}. Skip the next cell.")
elif not CREDENTIALS_PATH.exists():
    print(f"❌  Credentials not found at {CREDENTIALS_PATH}.")
    print("    Download a Desktop OAuth client from Google Cloud Console and save it there.")
else:
    print(f"📄 Credentials found. Run the next cell to authorize.")


In [ ]:
# Run this once. A browser window will open for you to authorize the app.
# After you click "Allow", a token file is written to TOKEN_PATH and you can skip this cell on future runs.
if not TOKEN_PATH.exists():
    sc.credentials_to_token(
        credentials_path=CREDENTIALS_PATH,
        output_path=TOKEN_PATH,
    )
    print(f"✅  Token written to {TOKEN_PATH}")
else:
    print(f"✅  Token already exists at {TOKEN_PATH} — nothing to do.")


---
## Step 4 — Log in as Data Owner

`sc.login_do(...)` initializes your Data Owner client using the token from Step 3.

In [ ]:
client = sc.login_do(email=DO_EMAIL, token_path=str(TOKEN_PATH))


---
## Step 5 — Approve incoming peer requests

A Data Scientist must request to connect with you before they can submit jobs. Pending requests appear in `client.peers` with state `requested_by_peer`.

> 💡 You can also pre-emptively add a peer with `client.add_peer(ds_email)` — useful if you're setting up a connection where the DS hasn't initiated yet.

In [ ]:
client.peers


Approve a specific Data Scientist by email. This is a one-time action per peer — once accepted, the connection persists across sessions.

In [ ]:
# ✏️ Approve this Data Scientist's request
DS_TO_APPROVE = DS_EMAILS[0]   # ✏️ or hard-code an email here

client.approve_peer_request(DS_TO_APPROVE)


---
## Step 6 — Create datasets

Each dataset has **two versions**:

- **Mock**: a synthetic look-alike with the same schema. Data Scientists see this freely and develop their analysis against it.
- **Private**: the real data. Stays on your machine; only results from approved jobs flow back to the DS.

`client.create_dataset(...)` accepts files OR folders for both `mock_path` and `private_path`. Use a folder when your dataset has multiple files (e.g. a CSV index plus companion PDFs).

We'll create two datasets to demonstrate both shapes:

| Dataset | Shape | Used for |
|---------|-------|----------|
| `beach_water_quality` | Single CSV | The example used in the DS tutorial |
| `lab_reports` | CSV index + PDFs | The multi-file pattern (the `load_index_with_companions` helper) |

### 6a — Simple dataset: a single CSV

Generates a small water-quality CSV with a mock and private version.

In [ ]:
import csv
import random
from datetime import date, timedelta
from pathlib import Path

DATA_DIR = Path("data").resolve()
DATA_DIR.mkdir(exist_ok=True)

def generate_beach_csv(out_path: Path, label: str, seed: int):
    """Generate a beach water quality CSV. Mock and private use different RNG seeds."""
    random.seed(seed)
    stations = [
        ("A1", "Sunset Beach North"),
        ("A2", "Sunset Beach South"),
        ("B1", "Harbor Point"),
        ("B2", "Harbor Cove"),
        ("C1", "Rocky Bay"),
    ]
    rows = []
    for i in range(90):
        station_id, station_name = random.choice(stations)
        d = date(2024, 1, 1) + timedelta(days=i)
        bacteria = max(0, int(random.gauss(120, 50)))
        rows.append({
            "date": d.isoformat(),
            "station_id": station_id,
            "station_name": station_name,
            "temperature_c": round(random.uniform(18, 25), 1),
            "ph": round(random.uniform(7.5, 8.5), 2),
            "salinity_ppt": round(random.uniform(30, 35), 1),
            "bacteria_cfu_ml": bacteria,
            "turbidity_ntu": round(random.uniform(2, 6), 1),
            "dissolved_oxygen_mg_l": round(random.uniform(8, 11), 1),
            "safe_to_swim": bacteria < 100,
            "inspector_id": f"{label}{i:03d}",
            "notes": "",
        })
    with open(out_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    return rows

mock_csv    = DATA_DIR / "beach_mock.csv"
private_csv = DATA_DIR / "beach_private.csv"
generate_beach_csv(mock_csv,    label="MOCK", seed=1)
generate_beach_csv(private_csv, label="REAL", seed=42)

print(f"✅  Generated {mock_csv}")
print(f"✅  Generated {private_csv}")


In [ ]:
# Create the dataset and share it with your Data Scientists.
# `users=DS_EMAILS` grants them read access to the mock + dataset metadata.
# `upload_private=False` (the default) keeps the private file local-only.

dataset = client.create_dataset(
    name="beach_water_quality",
    mock_path=mock_csv,
    private_path=private_csv,
    summary="Beach water quality readings (90 days, 5 stations)",
    tags=["water", "environmental"],
    users=DS_EMAILS,
)
print(f"✅  Dataset {dataset.name!r} created and shared with {DS_EMAILS}")


### 6b — Multi-file dataset: CSV index + PDFs

This is the dataset the DS tutorial's `load_index_with_companions` helper is designed for. The mock and private versions have the **same schema** (so DS code works in both places) but different content — the mock uses placeholder names, the private uses real-looking names.

> 💡 In a real deployment, your private data would already exist on disk. Here we synthesize it for demonstration.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

# Real-looking patient names (used only for the private version)
REAL_PATIENTS = [
    ("Alice Johnson", 34, "F"), ("Bob Smith", 52, "M"),
    ("Carol Davis", 28, "F"), ("David Wilson", 67, "M"),
    ("Eve Martinez", 45, "F"), ("Frank Lee", 39, "M"),
    ("Grace Chen", 31, "F"), ("Henry Park", 58, "M"),
    ("Iris Patel", 42, "F"), ("Jack Brown", 29, "M"),
]

def generate_lab_pdf(out_path: Path, name: str, age: int, sex: str,
                    report_date: date, glucose: float, cholesterol: float, hemoglobin: float):
    """Render a one-page lab report PDF."""
    c = canvas.Canvas(str(out_path), pagesize=letter)
    c.setFont("Helvetica-Bold", 14); c.drawString(72, 720, "Lakeside Clinic — Lab Report")
    c.setFont("Helvetica", 10)
    c.drawString(72, 700, f"Patient: {name}")
    c.drawString(72, 685, f"Age: {age}    Sex: {sex}")
    c.drawString(72, 670, f"Report date: {report_date}")
    c.line(72, 660, 540, 660)
    c.setFont("Helvetica-Bold", 11); c.drawString(72, 640, "Test Results")
    c.setFont("Helvetica", 10)
    c.drawString(90, 620, f"Glucose:      {glucose} mg/dL")
    c.drawString(90, 605, f"Cholesterol:  {cholesterol} mg/dL")
    c.drawString(90, 590, f"Hemoglobin:   {hemoglobin} g/dL")
    c.save()


def build_lab_dataset(out_dir: Path, *, use_real_names: bool, seed: int):
    """Generate index.csv + one PDF per row in `out_dir`. Returns the directory."""
    random.seed(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, (real_name, age, sex) in enumerate(REAL_PATIENTS, start=1):
        patient_id = f"P{i:03d}"
        # Mock uses placeholder names; private uses real-looking names
        name = real_name if use_real_names else f"Patient MOCK{i:03d}"
        report_date = date(2025, 6, 1) + timedelta(days=random.randint(0, 90))
        glucose = round(random.gauss(95, 15), 1)
        cholesterol = round(random.gauss(190, 30), 1)
        hemoglobin = round(random.gauss(14, 1.5), 1)
        report_filename = f"report_{patient_id}.pdf"

        generate_lab_pdf(out_dir / report_filename, name, age, sex,
                        report_date, glucose, cholesterol, hemoglobin)
        rows.append({
            "patient_id":         patient_id,
            "patient_name":       name,
            "age":                age,
            "sex":                sex,
            "report_date":        report_date.isoformat(),
            "glucose_mg_dl":      glucose,
            "cholesterol_mg_dl":  cholesterol,
            "hemoglobin_g_dl":    hemoglobin,
            "report_filename":    report_filename,
        })

    with open(out_dir / "index.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    return out_dir


lab_mock_dir    = DATA_DIR / "lab_mock"
lab_private_dir = DATA_DIR / "lab_private"
build_lab_dataset(lab_mock_dir,    use_real_names=False, seed=1)
build_lab_dataset(lab_private_dir, use_real_names=True,  seed=42)

print(f"✅  Mock    : {lab_mock_dir} ({len(list(lab_mock_dir.iterdir()))} files)")
print(f"✅  Private : {lab_private_dir} ({len(list(lab_private_dir.iterdir()))} files)")


Optional: write a README that's shared alongside the mock data. The DS sees this when they browse the dataset.

In [ ]:
lab_readme = DATA_DIR / "lab_readme.md"
lab_readme.write_text("""# Lab Reports Dataset

Each row in `index.csv` corresponds to a patient lab report. The PDF for that
record is in the same folder, named per the `report_filename` column.

## Columns

- `patient_id` — primary key, format `P###`
- `patient_name` — full name (mock data uses placeholder names)
- `age`, `sex` — demographics
- `report_date` — ISO date of the lab visit
- `glucose_mg_dl`, `cholesterol_mg_dl`, `hemoglobin_g_dl` — test values
- `report_filename` — companion PDF in the same folder

## Suggested DS analyses

- Aggregate statistics on test values
- Extract text from PDFs (e.g. with `pypdf`) and look for patterns
""")
print(f"✅  README written to {lab_readme}")


In [ ]:
dataset = client.create_dataset(
    name="lab_reports",
    mock_path=lab_mock_dir,         # folder → all files inside become mock_files
    private_path=lab_private_dir,
    summary="Patient lab reports — CSV index with companion PDFs",
    readme_path=lab_readme,
    tags=["medical", "pdf", "tabular"],
    users=DS_EMAILS,
)
print(f"✅  Dataset {dataset.name!r} created with {len(list(lab_mock_dir.iterdir()))} mock files")


### Verify what's been published

`client.datasets.get_all()` lists every dataset on this datasite (yours + any you've published).

In [ ]:
for ds in client.datasets.get_all():
    print(f"  • {ds.name}  —  {ds.summary or '(no summary)'}")


---
## Step 7 — List submitted jobs

Once a Data Scientist submits a job to you, it appears in `client.jobs`. The status reflects where it is in the lifecycle:

| Status | Meaning |
|--------|---------|
| 📥 `received` | Files have arrived but the runner hasn't validated them yet. |
| ⏳ `pending` | Validated and **waiting for your review**. |
| 🔄 `approved` | You approved it. Will run on the next call to `client.process_approved_jobs()`. |
| ⚙️ `running` | Currently executing on your machine. |
| ✅ `done` | Finished successfully. |
| 🚫 `rejected` | You rejected it. |
| ❌ `failed` | The DS's script crashed during execution. |

In [ ]:
emoji = {
    "received": "📥", "pending":  "⏳", "approved": "🔄",
    "running":  "⚙️",  "done":     "✅", "rejected": "🚫",
    "failed":   "❌",
}

if not client.jobs:
    print("📭 No jobs submitted yet.")
else:
    print(f"📋 Jobs ({len(client.jobs)} total):\n")
    for i, job in enumerate(client.jobs):
        icon = emoji.get(job.status, "❓")
        print(f"  [{i}]  {icon} {job.status:9s}  {job.name}  (from {job.submitted_by})")


---
## Step 8 — Review a job

Before approving, **read the source code** the Data Scientist submitted. This is the moment where you decide whether the script is safe to run on your private data.

In [ ]:
JOB_INDEX = 0   # ✏️ index from the list above

if JOB_INDEX < 0 or JOB_INDEX >= len(client.jobs):
    print(f"❌ JOB_INDEX={JOB_INDEX} is out of range. Pick 0..{len(client.jobs) - 1}.")
else:
    job = client.jobs[JOB_INDEX]
    print(f"Job        : [{JOB_INDEX}] {job.name}")
    print(f"Status     : {job.status}")
    print(f"Submitted  : {job.submitted_at}")
    print(f"Submitter  : {job.submitted_by}")
    print(f"\n{'─' * 60}")
    print(f"Code submitted (review carefully before approving):")
    print(f"{'─' * 60}\n")
    print(job.code)


### Approve or reject

If the code looks safe, call `job.approve()`. To reject, use `job.reject(reason="...")` — the reason is shared back to the DS.

In [ ]:
# Approve the job picked above (uncomment one of the lines below):

# job.approve()
# job.reject(reason="Script reads files outside the dataset; please scope access.")

print(f"Job is currently: {job.status!r}")


---
## Step 9 — Run approved jobs

Approving a job doesn't run it — `client.process_approved_jobs()` does. Each approved job runs in its own isolated Python virtual environment (the DS's `dependencies=[...]` are installed automatically).

### Sharing results with the submitter

By default, results stay on your machine. To send them back to the DS, pass `share_outputs_with_submitter=True`. Almost always what you want.

> 💡 Alternative: leave it `False` for jobs where you want to inspect the outputs first, then call `job.share_outputs([ds_email])` manually after review.

In [ ]:
client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,    # share stdout/stderr too — helps the DS debug
)


---
## Step 10 — Auto-approval for trusted job patterns

Reviewing every job by hand gets old fast for repeat workflows. **Auto-approval** lets you whitelist a specific `main.py` (matched by content hash) so future jobs with the same code run without manual review. The DS can change `params.json` between runs without invalidating the rule.

### How to register a rule

The cleanest way is to take a *pending* job you trust and turn it into a rule:

```python
import syft_bg
syft_bg.auto_approve_job(
    job,                          # a JobInfo from client.jobs
    file_paths=["params.json"],   # files matched by NAME only (params can change)
    # contents (the rest)         # files matched by NAME + CONTENT (locked-down)
    # peers=[ds_email]            # restrict to a specific DS (optional)
)
```

After registering, the `approve` background service from Step 11 picks up the rule and approves matching jobs automatically.

In [ ]:
import syft_bg

# Pick a pending job to use as the auto-approval template
JOB_INDEX_FOR_AUTO = 0   # ✏️ index from the Step 7 list

candidate = client.jobs[JOB_INDEX_FOR_AUTO]
print(f"Using as template: {candidate.name!r} (status: {candidate.status})")
print(f"Files in submission:")
for f in candidate.files:
    print(f"   • {f.name}")


In [ ]:
# Register the rule. main.py is content-hashed; params.json matches by name only.
# (Adjust file_paths if the job uses different file names.)

result = syft_bg.auto_approve_job(
    candidate,
    file_paths=["params.json"],   # ✏️ name-matched files (parameters that can change)
)
if result.success:
    print(f"✅  Rule registered as {result.name!r}")
else:
    print(f"❌  Failed: {result.error}")


---
## Step 11 — Run unattended with `syft_bg`

Up to now you've been doing each step manually. For a real deployment you'll want background services that:

- 🔄 **Sync** — keep your SyftBox in sync with Drive so new submissions appear automatically.
- 📧 **Notify** — email you (or the DS) when peer requests / new jobs / completed jobs occur.
- 🤖 **Approve** — apply auto-approval rules from Step 10 without you in the loop.
- 🏃 **Run approved jobs** — execute approved jobs as they appear.

`syft_bg.init(...)` writes the configuration; `syft_bg.start()` launches the daemons.

> ⚠️ Background services are **persistent processes** — they keep running after this notebook closes. Use `syft_bg.stop()` to shut them down (Step 13).

In [ ]:
syft_bg.init(
    email=DO_EMAIL,
    drive_token_path=str(TOKEN_PATH),
    notify_jobs=True,         # email me about new jobs
    notify_peers=True,        # email me about peer requests
    approve_jobs=True,        # apply auto-approval rules
    approve_peers=False,      # don't auto-approve peer requests (security default)
    notify_interval=30,       # seconds
    approve_interval=5,       # seconds
)


Start the services. `ensure_running()` is the polite version that starts anything not already running.

In [ ]:
results = syft_bg.ensure_running()
for service, (success, message) in results.items():
    icon = "✅" if success else "❌"
    print(f"  {icon} {service:10s}  {message}")


---
## Step 12 — Inspect service logs

If something looks off — jobs not getting picked up, emails not arriving — `syft_bg.logs(service, n=20)` shows the most recent log lines for that service.

In [ ]:
SERVICE = "approve"   # ✏️ one of: "sync", "notify", "approve"
N_LINES = 20

for line in syft_bg.logs(SERVICE, n=N_LINES):
    print(line)


---
## Step 13 — Cleanup

When you're done, stop the background services and (optionally) remove datasets you no longer want to host.

In [ ]:
# Stop all background services
results = syft_bg.stop()
for service, (success, message) in results.items():
    print(f"  {'✅' if success else '❌'} {service:10s}  {message}")


In [ ]:
# Remove a dataset you no longer want to host (uncomment to use):
# client.delete_dataset("lab_reports")
# client.delete_dataset("beach_water_quality")


---
## 🎉 You're done!

You've successfully:
- ✅ Logged in as a Data Owner
- ✅ Approved an incoming peer request
- ✅ Created a single-file dataset and a CSV-with-PDFs dataset
- ✅ Reviewed and approved a Data Scientist's job
- ✅ Ran the job and shared the results back
- ✅ Set up an auto-approval rule for a trusted job pattern
- ✅ Started background services for unattended operation

## Day-to-day workflow

Once initial setup (Steps 1–6, 11) is done, ongoing operation is mostly hands-off:

```
syft_bg.ensure_running()  # if you restart your machine
# Receive emails when new jobs arrive
# Reply to email or call job.approve() in this notebook
client.process_approved_jobs(share_outputs_with_submitter=True)
```

## Resources

- [Data Owner guide](https://www.mintlify.com/OpenMined/syft-client/guides/data-owner)
- [`syft-bg` background services docs](https://www.mintlify.com/OpenMined/syft-client/packages/syft-bg)
- [Full API reference](https://www.mintlify.com/OpenMined/syft-client/api/client/login)
- [OpenMined Community Slack](https://slack.openmined.org)
